# CS2 (국내반) — Lecture 3 옵셔널 Lab: 다차원 리스트 × 데이터사이언스

**이 노트북은 강의 진도·시험 범위와 무관한 옵셔널 Lab이다.** 채점하지 않는다.
목적은 딱 하나 — Lecture 3에서 익힌 *"다차원 리스트를 중첩 for 루프로 훑기"*
가 데이터사이언스 도구(numpy 배열, pandas 표, 이미지, 히트맵)에서 그대로
재등장한다는 걸 눈으로 보는 것.

- 이미지 = `높이 × 너비 × 3` 리스트, 표 = `행 × 열` 리스트, 시계열 =
  `시간 × 종목` 리스트 — 전부 Lecture 3에서 다룬 다차원 리스트다.
- Colab에는 `numpy`, `matplotlib`, `pandas`, `scikit-learn` 이 이미 있다.
  로컬이면 `pip install numpy matplotlib pandas scikit-learn`.
- **Lab D는 인터넷에서 주가 CSV를 받아온다.** 실패하면 자동으로 합성
  데이터로 넘어가니 셀은 항상 돌아간다.
- 각 Lab의 준비 코드와 그림은 채워져 있다. 채울 곳은
  `# ADD ADDITIONAL CODE HERE!` 뿐이고, 그 부분은 **순수 리스트 + for
  루프**로 구현한다. numpy/pandas는 정답 대조와 시각화에만 쓴다.

| Lab | 내용 | Lecture 3 연결 |
|---|---|---|
| A | RGB 이미지 → grayscale | Problem 2·3 (3차원 배열 순회) |
| B | box blur (이미지 흐리기) | Problem 6 (3×3 이웃 순회) |
| C | 흰 배경에서 객체만 crop | Problem 3 (max/min), Problem 7 (`maxZeroRect`) |
| D | 여러 종목 주가 시계열 | Problem 4 (행/열 순회), 이동평균 = 1D blur |
| E | 두 종목 수익률 2D 히스토그램 | Problem 6 (격자 만들기) |

## A. 이미지 = 높이 × 너비 × 3 짜리 3차원 리스트

컬러 이미지 한 장은 `img[i][j] = [R, G, B]` 꼴의 `H×W×3` 3차원 리스트다
(Problem 2·3의 3차원 배열과 같은 모양). 흑백으로 바꾸는 표준 공식은
$gray = 0.299\,R + 0.587\,G + 0.114\,B$.

Write a function `rgb2gray`:
- input parameter: a 3-dimensional list `img` of shape `H×W×3`
  (each pixel `[R, G, B]`, integers `0`–`255`)
- return value: a 2-dimensional list `gray` (`H×W`) where
  `gray[i][j] = round(0.299*R + 0.587*G + 0.114*B)`

Problem 2·3처럼 바깥 2중 for 루프로 픽셀을 돌고 안에서 채널 3개로 가중합.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

img = load_sample_image("china.jpg").tolist()   # H x W x 3 파이썬 리스트
H, W = len(img), len(img[0])
print(H, W, len(img[0][0]))                      # 427 640 3

def rgb2gray(img):
    H, W = len(img), len(img[0])
    gray = [[0] * W for _ in range(H)]
    # ADD ADDITIONAL CODE HERE!

    return gray

gray = rgb2gray(img)

# --- numpy로 정답 대조 (max error <= 1 이면 정상) ---
ref = np.array(img) @ np.array([0.299, 0.587, 0.114])
print("max error:", np.abs(np.array(gray) - ref).max())

# --- 시각화 ---
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.array(img, dtype=np.uint8)); ax[0].set_title("RGB (3D list)")
ax[1].imshow(np.array(gray), cmap="gray");   ax[1].set_title("gray (2D list)")
for a in ax:
    a.axis("off")
plt.show()

## B. box blur — Problem 6의 이웃 순회 그대로

Problem 6에서는 각 칸의 3×3 이웃에 있는 지뢰 **개수**를 셌다. 개수 대신
**평균**을 내면 이미지가 흐려진다(box blur — 카메라 필터, CNN의 기본형).

Write a function `blur`:
- input parameter: a 2-dimensional list `g` (`H×W`)
- return value: same-shape 2-dimensional list, 각 칸을 자기 자신과 격자 안에
  있는 이웃(최대 8개)의 **평균**으로 바꾼 것
  - 경계 칸은 격자 안에 있는 이웃만 (Problem 6의 `withinBoundary`와 동일)

Problem 6 코드에서 `mines[i][j] += 1` 을 "이웃 값 합산 + 이웃 수 세기 →
마지막에 나누기" 로만 바꾸면 된다.

In [ ]:
def withinBoundary(H, W, i, j):
    return 0 <= i < H and 0 <= j < W

def blur(g):
    H, W = len(g), len(g[0])
    out = [[0.0] * W for _ in range(H)]
    # ADD ADDITIONAL CODE HERE!

    return out

# --- 작은 격자로 손검증 ---
small = [[0, 0, 0, 0],
         [0, 9, 9, 0],
         [0, 9, 9, 0],
         [0, 0, 0, 0]]
print(blur(small)[1][1])   # (9+9+9+9+0+0+0+0+0) / 9 = 4.0

# --- 시각화: 일부만 잘라서 여러 번 blur ---
g0 = [row[220:420] for row in gray[120:280]]     # 160 x 200
b = g0
for _ in range(4):
    b = blur(b)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.array(g0), cmap="gray"); ax[0].set_title("original crop")
ax[1].imshow(np.array(b), cmap="gray");  ax[1].set_title("blur x4")
for a in ax:
    a.axis("off")
plt.show()

## C. 흰 배경에서 객체만 잘라내기 (bounding-box crop)

배경이 흰색인 사진에서 물체가 차지하는 **가장 작은 직사각형**만 남기고
잘라내는 함수. 흰색이 아닌 픽셀들의 행 인덱스 최소·최대, 열 인덱스
최소·최대를 찾으면 그게 곧 그 직사각형이다 — Problem 3 `maxPrime`의 max/min
순회를 네 값에 대해 동시에 돌리는 것. (Problem 7 `maxZeroRect`의 사촌.)

Write a function `crop_object`:
- input parameter: a 3-dimensional list `img` (`H×W×3`, `0`–`255`) and an
  integer `thresh` (default `250`)
  - 픽셀이 "배경(흰색)"이라는 건 `R`, `G`, `B` 가 **모두** `thresh` 이상
- return value: a 3-dimensional list — 흰색이 아닌 픽셀을 모두 포함하는 가장
  작은 직사각형으로 자른 부분 이미지
  - 흰색이 아닌 픽셀이 하나도 없으면 `None`

풀이 순서: (1) 2중 for 루프로 전체를 훑으며 흰색 아닌 픽셀마다 `min_i,
max_i, min_j, max_j` 갱신 (2) 그 범위를 for 루프로 새 리스트에 복사
(Problem 6에서 격자 만들던 방식).

셀 출력에서, 우리가 for 루프로 한 것과 **numpy 슬라이싱** `a[i1:i2, j1:j2]`
한 줄이 같은 결과임을 확인한다. numpy 배열은 축마다 `start:stop:step` 을
콤마로 나열해서(`a[행, 열, 채널]`) 부분 배열을 바로 꺼낸다 — 리스트로는
중첩 for 루프로 짜야 하던 일이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 흰 배경 + 객체 2개(주황 원, 파란 사각형) 합성 이미지 ---
Himg, Wimg = 200, 300
arr = np.full((Himg, Wimg, 3), 255, dtype=int)
yy, xx = np.ogrid[:Himg, :Wimg]
arr[(yy - 60)**2 + (xx - 210)**2 <= 45**2] = [255, 140, 0]
arr[120:170, 40:110] = [30, 90, 200]
img = arr.tolist()                     # H x W x 3 리스트 (배경 255)

def is_white(px, thresh):
    return px[0] >= thresh and px[1] >= thresh and px[2] >= thresh

def crop_object(img, thresh=250):
    H, W = len(img), len(img[0])
    min_i = min_j = 10**9
    max_i = max_j = -1
    # ADD ADDITIONAL CODE HERE!
    #   흰색 아닌 픽셀마다 min_i, max_i, min_j, max_j 갱신 (Problem 3 max/min 패턴)
    #   흰색 아닌 픽셀이 하나도 없으면  return None

    out = []                           # 찾은 직사각형을 새 리스트로 복사
    for i in range(min_i, max_i + 1):
        out.append([img[i][j] for j in range(min_j, max_j + 1)])
    return out

crop = crop_object(img)

if not crop or not crop[0]:
    print("crop_object 미완성 — min/max 스캔을 채우세요")
else:
    print("원본", len(img), "x", len(img[0]), " ->  crop", len(crop), "x", len(crop[0]))

    # --- 같은 일을 numpy 슬라이싱으로: 위 2중 for 루프 = 한 줄 ---
    a = np.array(img)
    nw = ~(a >= 250).all(axis=2)               # 흰색이 아닌 곳
    ys, xs = np.where(nw)
    i1, i2 = ys.min(), ys.max() + 1
    j1, j2 = xs.min(), xs.max() + 1
    crop_np = a[i1:i2, j1:j2]                  # [행 구간, 열 구간, 채널 전체]
    print("numpy 슬라이싱 결과와 일치:", np.array_equal(np.array(crop), crop_np))
    print("  a.shape                =", a.shape)
    print("  a[i1:i2, j1:j2].shape  =", a[i1:i2, j1:j2].shape, " (= crop)")
    print("  a[:, :, 0].shape       =", a[:, :, 0].shape, " (R 채널만 -> 2D)")
    print("  a[::20, ::20].shape    =", a[::20, ::20].shape, " (20px 간격 다운샘플)")

    # --- 시각화 ---
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    ax[0].imshow(a.astype(np.uint8)); ax[0].set_title("original (white bg)")
    ax[0].add_patch(plt.Rectangle((j1, i1), j2 - j1, i2 - i1, fill=False, ec="lime", lw=2))
    ax[1].imshow(np.array(crop, dtype=np.uint8)); ax[1].set_title("crop_object(img)")
    for ax_ in ax:
        ax_.axis("off")
    plt.show()

## D. 여러 종목 주가 = 시간 × 종목 2차원 리스트

`pandas`로 인터넷에서 주가 CSV를 받아 `M[t][k]` = `t`일차 `k`번 종목의
종가인 2차원 리스트를 만든다. 시간축(행)으로 훑느냐 종목축(열)으로 훑느냐가
Problem 4의 두 방향 순회고, 이동평균은 시간축 이웃 평균 = **1차원 blur**(Lab B)다.

Write a function `normalize`:
- input parameter: a 2-dimensional list `M` (`days × k`), 각 열이 한 종목
- return value: same-shape 2-dimensional list — 각 열을 그 열의 **첫 행
  값**으로 나눈 것 (첫날 = 1.0 기준 상대가격)

Write a function `moving_avg`:
- input parameter: a 1-dimensional list `col` and a window size `w`
- return value: same-length list where `out[t]` = `col[max(0, t-w+1) .. t]`
  의 평균 (앞쪽이 모자라면 있는 것만 평균)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKERS = ["AAPL", "MSFT", "IBM"]
URL = "https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv"

try:
    df = pd.read_csv(URL)
    px = df[["Date"] + TICKERS].dropna()
    px = px[(px["Date"] >= "2012-01-01") & (px["Date"] < "2014-01-01")]
    px = px.sort_values("Date").reset_index(drop=True)
    assert len(px) > 100
except Exception as e:
    print("다운로드 실패 -> 합성 데이터 사용:", repr(e))
    rng = np.random.default_rng(3)
    n = 250
    walk = rng.normal(0.0004, 0.02, size=(n, len(TICKERS))).cumsum(axis=0)
    px = pd.DataFrame(100 * np.exp(walk), columns=TICKERS)
    px.insert(0, "Date", pd.date_range("2012-01-03", periods=n, freq="B").astype(str))

print(px.shape)
print(px.head())
M = px[TICKERS].values.tolist()      # days x 종목  2차원 리스트

def normalize(M):
    days, k = len(M), len(M[0])
    out = [[0.0] * k for _ in range(days)]
    # ADD ADDITIONAL CODE HERE!

    return out

def moving_avg(col, w):
    out = [0.0] * len(col)
    # ADD ADDITIONAL CODE HERE!

    return out

norm = normalize(M)
col0 = [row[0] for row in M]

# --- pandas 결과와 대조 ---
ref_norm = px[TICKERS] / px[TICKERS].iloc[0]
print("normalize max error:", np.abs(np.array(norm) - ref_norm.values).max())
ref_ma = px[TICKERS[0]].rolling(20, min_periods=1).mean().tolist()
print("moving_avg max error:", max(abs(a - b) for a, b in zip(moving_avg(col0, 20), ref_ma)))

# --- 시각화 ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, t in enumerate(TICKERS):
    ax[0].plot([row[k] for row in norm], label=t)
ax[0].axhline(1, lw=.6, color="k"); ax[0].legend(); ax[0].set_title("normalized (day 0 = 1.0)")
ax[1].plot(col0, lw=.8, label=TICKERS[0])
ax[1].plot(moving_avg(col0, 20), label="20d moving avg")
ax[1].legend(); ax[1].set_title("price vs moving average")
plt.show()

## E. 두 종목 일간수익률의 2D 히스토그램 (= Problem 6의 격자 만들기)

`R[t][k] = M[t+1][k] / M[t][k] - 1` (전날 대비 수익률). 두 종목의 그 날
수익률을 점 `(x, y)` 로 찍으면 산점도가 되고, 평면을 격자로 나눠 칸마다 점
개수를 세면 두 종목이 **같이 움직이는지**(상관)가 히트맵으로 보인다. 격자
생성·채우기는 Problem 6에서 `mines` 격자를 만들던 것과 똑같다.

Write a function `daily_returns`:
- input parameter: a 2-dimensional list `M` (`days × k`)
- return value: a 2-dimensional list `R` of shape `(days-1) × k` where
  `R[t][k] = M[t+1][k] / M[t][k] - 1`

Write a function `hist2d`:
- input parameter: `pts` (`[[x, y], ...]`), `nbins`, `lo`, `hi`
  (x·y 공통 범위)
- return value: an `nbins × nbins` integer list `Hgrid` where `Hgrid[r][c]`
  = 그 칸의 점 개수
  - 칸: `c = int((x - lo) / (hi - lo) * nbins)`, `r` 도 같은 식(y로)
  - 인덱스가 `nbins`면 `nbins - 1`로, 범위 밖 점은 버린다

In [ ]:
def daily_returns(M):
    days, k = len(M), len(M[0])
    R = [[0.0] * k for _ in range(days - 1)]
    # ADD ADDITIONAL CODE HERE!

    return R

def hist2d(pts, nbins, lo, hi):
    Hgrid = [[0] * nbins for _ in range(nbins)]
    # ADD ADDITIONAL CODE HERE!

    return Hgrid

R = daily_returns(M)
pts = [[row[0], row[1]] for row in R]     # (AAPL 수익률, MSFT 수익률)

nbins, lo, hi = 15, -0.06, 0.06
Hgrid = hist2d(pts, nbins, lo, hi)
print("담긴 점:", sum(sum(r) for r in Hgrid), "/", len(pts))

# --- numpy 결과와 대조 ---
ref, _, _ = np.histogram2d([p[0] for p in pts], [p[1] for p in pts],
                           bins=nbins, range=[[lo, hi], [lo, hi]])
print("일치하는 칸:", int((np.array(Hgrid).T == ref.astype(int)).sum()), "/", nbins * nbins)

# --- 산점도 vs 손으로 만든 히트맵 ---
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter([p[0] for p in pts], [p[1] for p in pts], s=12)
ax[0].axhline(0, lw=.5); ax[0].axvline(0, lw=.5)
ax[0].set_xlabel(TICKERS[0] + " daily return")
ax[0].set_ylabel(TICKERS[1] + " daily return")
ax[0].set_title("scatter")
ax[1].imshow(Hgrid, origin="lower", extent=[lo, hi, lo, hi])
ax[1].set_title("hist2d (our 2D list)")
plt.show()